# Dimensionality Reduction
- Speed up training
- Possibly imporve model performance
- Useful for data visualization


## PCA Principal Component Analysis
Identifies the axis that accounts for the largest amount of variance in the training set. It also finds a second axis, orthogonal to the first one that
accounts for the largest amount of remaining variance, and so on in higher dimensions.
If you use PCA as a preprocessing step before a model, make sure you always retrain the model entirely every time you update the PCA transformer.  
SVD:
    $$ \mathbf{X}=\mathbf{U\Sigma V^T} $$
Use SVD to find the principal components. Once you've identified all the principal components,  you reduce the dimensionality to $d$ dimensions.
$$ \mathbf{X_{d-proj}} = \mathbf{XW_d}$$ 
where $\mathbf{W_d}$ is the matrix containing the first d columns of $\mathbf{V}$ 
 

In [1]:
import numpy as np

In [4]:
X= np.random.rand(3,4,3)
X_centered = X - X.mean(axis=0)
U,s, Vt = np.linalg.svd(X_centered)  #always remember to center the data
Vt[0]

array([[-0.09397894,  0.59676911, -0.79689058],
       [-0.91140352, -0.37368069, -0.17235534],
       [ 0.40063896, -0.71009111, -0.57901558]])

In [5]:
Vt

array([[[-0.09397894,  0.59676911, -0.79689058],
        [-0.91140352, -0.37368069, -0.17235534],
        [ 0.40063896, -0.71009111, -0.57901558]],

       [[ 0.87261687,  0.41149755, -0.26307711],
        [-0.45838715,  0.87594803, -0.15032054],
        [ 0.16858535,  0.2517634 ,  0.95299222]],

       [[-0.73894984,  0.2594536 , -0.62180139],
        [-0.64781457, -0.01995718,  0.7615366 ],
        [-0.18517401, -0.96554935, -0.18282518]]])

In [7]:
W2 = Vt[:2].T
X2D = X_centered @ W2
X2D

array([[[-0.0619142 ,  0.2407388 ],
        [ 0.14799111,  0.36201482],
        [ 0.38687046, -0.00573672],
        [-0.3571055 , -0.1459829 ]],

       [[-0.21644996, -0.34098802],
        [ 0.09427322,  0.00228897],
        [ 0.35104155,  0.33808938],
        [ 0.1455926 , -0.19211092]],

       [[-0.0615999 , -0.15027908],
        [ 0.62355795, -0.28564961],
        [ 0.1735022 ,  0.0891891 ],
        [-0.12289708,  0.34936624]]])

In [10]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X2D = pca.fit_transform(X[0])
X2D

array([[ 0.28729871,  0.01199035],
       [ 0.35958859,  0.16828351],
       [-0.5626232 ,  0.16545783],
       [-0.0842641 , -0.34573169]])

In [12]:
pca.explained_variance_ratio_

array([0.75090192, 0.24591602])

In [ ]:
from sklearn.datasets import fetch_openml

mnist= fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

pca =PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
#d =np.argmax(cumsum >=0.95) + 1


In [21]:
d=np.argmax(cumsum >=0.95) + 1

In [23]:
#pca=PCA(n_components=d)

#better way, can specify the ratio of variance you want to preserve
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [24]:
pca.n_components_

np.int64(154)

In [25]:
#Alternatively we can tune the number of dimensions as we would any other hyperparameter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42), RandomForestClassifier(random_state=42))
param_distrib = {
    "pca__n_components": np.arange(10,80),
    "randomforestclassifier__n_estimators": np.arange(50,500)
}
rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'pca__n_components': array([10, 11... 78, 79]), 'randomforestclassifier__n_estimators': array([ 50, ...97, 498, 499])}"
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yield

In [26]:
rnd_search.best_params_

{'randomforestclassifier__n_estimators': np.int64(314),
 'pca__n_components': np.int64(36)}

We can calculate the _reconstruction error_ which is the mean squared distance btw the original data and the  
compressed data, first by recovering the original data:

In [27]:
x_recovered = pca.inverse_transform(X_reduced)

#### Randomized PCA

In [28]:
rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)

### Incremental PCA
Allows you to split the training set into batches so you don't have to fit it all in memory.

In [30]:
from sklearn.decomposition import IncrementalPCA

n_batches=100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)
X_reduced = inc_pca.transform(X_train)

Alternatively can use numpys memmap to memory map file on disk:


In [32]:
filename="mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='write', shape= X_train.shape)
X_mmap[:] = X_train # or use for loop like above to load each batch of X_train to X_mmap
X_mmap.flush()   #flush to ensure any data still in the cache gets saved to disk
del X_mmap

In [33]:
#Load file and memory map it
X_mmap = np.memmap(filename, dtype="float32", mode="readonly").reshape(-1,784)  
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)

,"n_components n_components: int, default=NoneNumber of components to keep. If ``n_components`` is ``None``,then ``n_components`` is set to ``min(n_samples, n_features)``.",154
,"batch_size batch_size: int, default=NoneThe number of samples to use for each batch. Only used when calling``fit``. If ``batch_size`` is ``None``, then ``batch_size``is inferred from the data and set to ``5 * n_features``, to provide abalance between approximation accuracy and memory consumption.",600
,"whiten whiten: bool, default=FalseWhen True (False by default) the ``components_`` vectors are dividedby ``n_samples`` times ``components_`` to ensure uncorrelated outputswith unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimesimprove the predictive accuracy of the downstream estimators bymaking data respect some hard-wired assumptions.",False
,"copy copy: bool, default=TrueIf False, X will be overwritten. ``copy=False`` can be used tosave memory but is unsafe for general use.",True
Name,Type,Value
batch_size_ batch_size_: intInferred batch size from ``batch_size``.,int,600
"components_ components_: ndarray of shape (n_components, n_features)Principal axes in feature space, representing the directions ofmaximum variance in the data. Equivalently, the right singularvectors of the centered input data, parallel to its eigenvectors.The components are sorted by decreasing ``explained_variance_``.","ndarray[float64](154, 784)","[[-0., 0.,-0.,..., 0., 0., 0.], [-0.,-0., 0.,...,-0.,-0.,-0.], [-0., 0., 0.,...,-0.,-0.,-0.], ..., [-0., 0.,-0.,...,-0.,-0.,-0.], [-0., 0.,-0.,...,-0.,-0.,-0.], [ 0., 0.,-0.,..., 0., 0., 0.]]"
"explained_variance_ explained_variance_: ndarray of shape (n_components,)Variance explained by each of the selected components.","ndarray[float64](154,)","[332724.64,243283.89,211507.31,..., 1448.24, 1407.4 , 1399.21]"
"explained_variance_ratio_ explained_variance_ratio_: ndarray of shape (n_components,)Percentage of variance explained by each of the selected components.If all components are stored, the sum of explained variances is equalto 1.0.","ndarray[float64](154,)","[0.1 ,0.07,0.06,...,0. ,0. ,0. ]"
"mean_ mean_: ndarray of shape (n_features,)Per-feature empirical mean, aggregate over calls to ``partial_fit``.","ndarray[float64](784,)","[0.,0.,0.,...,0.,0.,0.]"
n_components_ n_components_: intThe estimated number of components. Relevant when``n_components=None``.,int,154


## Random Projection
PCA (and even randomized PCA) might be too slow if you have tens of thousands of features.
Calculate $d$ the _johnson-lindestrauss_ statistics to determine the new dimension for features then create a random matrix, $\mathbf{P}$ of dimension $[n,d]$
and use it project a new reduced dimension matrix $[m,d]$

In [34]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim
m, ε = 5_000, 0.1   #m is the size of the dataset, ε is the max distance we want between 2 instances after dimensionality reduction
d = johnson_lindenstrauss_min_dim(m, eps=ε)
d

np.int64(7300)

In [36]:
n=20_000
rng = np.random.default_rng(seed=42)

#we generate a random matrix P of shape [d,n] with mean 0 and variance 1/d and use it to project a dataset of m,n dimensions down to m,d
P= rng.standard_normal((d,n)) / np.sqrt(d)
X= rng.standard_normal((m,n)) #generate a fake dataset to illustrate
X_reduced = X @ P.T

In [37]:
#sklearn has Gaussian RandomProjection to do the above
from sklearn.random_projection import GaussianRandomProjection

gaussian_rnd_proj = GaussianRandomProjection(eps=ε, random_state=42)
X_reduced = gaussian_rnd_proj.fit_transform(X)

In [38]:
from sklearn.random_projection import SparseRandomProjection
sparse_rnd_proj = SparseRandomProjection(eps=0.1, random_state=42)
X_reduced = sparse_rnd_proj.fit_transform(X)

In [39]:
sparse_rnd_proj.components_

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1035298 stored elements and shape (7300, 20000)>

In [1]:
#To recover the inverse:
#components_pinv = np.linalg.pinv(gaussian_rnd_proj.components_)
#X_recovered = X_reduced @ components_pinv.T